# Regime-dependent momentum and reversal

**Hypothesis:** momentum works in normal conditions, and short-term reversal works in
crisis conditions (high volatility with high volume).

**Regimes (point-in-time, so no look-ahead)**
* **Volatility:** average 20-day realized volatility across coins, measured at each Friday.
  High volatility means it is above its *expanding* 75th percentile, computed only from past
  weeks.
* **Volume:** total USD volume that week divided by its trailing 26-week median. High volume
  means the ratio is above 1.
* **Crisis:** high volatility and high volume in the same week. The decision made at the close
  of week t is applied to week t+1.

**Strategies compared (all weekly, all net of 20 bps)**
1. Momentum in all weeks (the base strategy).
2. Momentum in normal weeks, flat in crisis weeks.
3. Reversal in crisis weeks only, flat otherwise.
4. Switch: momentum in normal weeks, reversal in crisis weeks.

Earlier versions used full-sample thresholds (look-ahead) and daily data, which added noise.
Every number is printed by code.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cryptolib import *  # noqa: F401,F403

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

UNIVERSE = ["BTCUSDT", "ETHUSDT", "ADAUSDT", "BNBUSDT", "XRPUSDT", "DOTUSDT", "MATICUSDT"]
START = "2020-01-01"
END = None
MOM_FORMATION_WEEKS = 2     # fixed a priori
REV_FORMATION_WEEKS = 1     # fixed a priori
VOL_WINDOW_DAYS = 20
VOL_QUANTILE = 0.75
MIN_WEEKS_FOR_THRESHOLD = 52
VOLUME_BASELINE_WEEKS = 26

## 1. Data

In [ ]:
close_d, qvol_d = load_universe(UNIVERSE, START, END)
px_d = common_window(close_d)
qv_d = qvol_d.loc[px_d.index[0]:px_d.index[-1], px_d.columns]
px_w = to_weekly(px_d)
ret_w = px_w.pct_change().dropna()
print(f"Common daily window: {px_d.index[0].date()} -> {px_d.index[-1].date()}")
print(f"Weekly returns: {len(ret_w)}")

## 2. Regime indicators (point-in-time)

In [ ]:
ret_d = px_d.pct_change()
rv_d = ret_d.rolling(VOL_WINDOW_DAYS).std().mean(axis=1)          # cross-coin average
rv_w = rv_d.resample("W-FRI").last().reindex(ret_w.index)
vol_thr = rv_w.expanding(min_periods=MIN_WEEKS_FOR_THRESHOLD).quantile(VOL_QUANTILE)
high_vol = (rv_w > vol_thr)

usd_vol_w = qv_d.sum(axis=1).resample("W-FRI").sum().reindex(ret_w.index)
vol_ratio = usd_vol_w / usd_vol_w.rolling(VOLUME_BASELINE_WEEKS,
                                          min_periods=VOLUME_BASELINE_WEEKS // 2).median()
high_volume = vol_ratio > 1.0

crisis = (high_vol & high_volume).fillna(False).astype(bool)
regime_table = pd.DataFrame({
    "Weeks with data": [int(rv_w.notna().sum())],
    "High-vol weeks": [int(high_vol.fillna(False).sum())],
    "High-volume weeks": [int(high_volume.fillna(False).sum())],
    "Crisis weeks": [int(crisis.sum())],
    "Crisis share": [crisis.mean()],
}, index=["count"]).T
print(regime_table.to_string())

## 3. Build the four strategies

In [ ]:
mom_w = rank_weights(ret_w.rolling(MOM_FORMATION_WEEKS, min_periods=MOM_FORMATION_WEEKS).mean())
rev_w = rank_weights(-ret_w.rolling(REV_FORMATION_WEEKS, min_periods=REV_FORMATION_WEEKS).mean())

calm = (~crisis).astype(float)
crisis_f = crisis.astype(float)
switch_arr = np.where(crisis.values[:, None], rev_w.values, mom_w.values)
switch_w = pd.DataFrame(switch_arr, index=ret_w.index, columns=ret_w.columns)

weights = {
    "1. Momentum, all weeks": mom_w,
    "2. Momentum, flat in crisis": mom_w.mul(calm, axis=0),
    "3. Reversal, crisis only": rev_w.mul(crisis_f, axis=0),
    "4. Switch (mom normal / rev crisis)": switch_w,
}
bts = {name: backtest(w, ret_w) for name, w in weights.items()}

## 4. Results, net of costs

In [ ]:
net_table = metrics_table({k: v["net"] for k, v in bts.items()}, PPY_WEEKLY)
print(net_table.to_string())
print()
print(pd.DataFrame({k: cost_summary(v, PPY_WEEKLY) for k, v in bts.items()}).to_string())

## 5. Does switching beat plain momentum?
HAC tests on the weekly difference from the base momentum strategy. The crisis sample is
small, so a large p-value is the expected outcome unless the effect is very strong.

In [ ]:
base_net = bts["1. Momentum, all weeks"]["net"]
rows = []
for name in ["2. Momentum, flat in crisis", "3. Reversal, crisis only",
             "4. Switch (mom normal / rev crisis)"]:
    d = mean_diff_test(bts[name]["net"], base_net)
    rows.append({"Strategy": name,
                 "Mean diff vs base (bps/wk)": d["mean diff"] * 1e4,
                 "t (HAC)": d["t-stat"],
                 "p-value": d["p-value"],
                 "n": d["n"]})
print(pd.DataFrame(rows).set_index("Strategy").to_string())

## 6. Performance inside each regime
What each strategy earns in normal weeks and in crisis weeks, with the number of weeks.

In [ ]:
rows = {}
for name, bt in bts.items():
    r = bt["net"]
    rows[name] = {"Normal-week mean (bps)": r[~crisis].mean() * 1e4,
                  "Crisis-week mean (bps)": r[crisis].mean() * 1e4,
                  "Normal weeks": int((~crisis & r.notna()).sum()),
                  "Crisis weeks": int((crisis & r.notna()).sum())}
print(pd.DataFrame(rows).T.to_string())

## 7. Chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for name, bt in bts.items():
    (1 + bt["net"].fillna(0)).cumprod().plot(ax=ax, label=name)
ax.set_yscale("log")
ax.set_title("Growth of $1, weekly, net of 20 bps")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("regime_switching_equity.png", dpi=150)
plt.show()

## 8. How to read the results
* The switch helps only if it beats plain momentum **net of costs** (section 4) and the
  difference is significant (section 5), not just if it has a higher Sharpe in one period.
* The reversal leg depends on very few crisis weeks (section 3), so its estimates are uncertain.
* Thresholds are set from past data only, so this is a fair test. Changing VOL_QUANTILE or
  the volume rule and then picking the best version would be data mining. Report any such
  sensitivity check as a sensitivity check, not as a result.

In [ ]:
print("Summary (all numbers computed above):")
sh = net_table.loc["Sharpe"]
print(f"  Momentum, all weeks  net Sharpe: {sh.iloc[0]:.2f}")
print(f"  Switch               net Sharpe: {sh.iloc[3]:.2f}")
print(f"  Crisis weeks: {int(crisis.sum())} of {int(rv_w.notna().sum())}")